# WHFast versus hierarchical-Jacobi WHFast

This notebook compares three REBOUND integrators on thirteen deliberately different hierarchical systems:

- **WHFast** with its ordinary left-deep Jacobi chain;
- **WHFast-HJ** with an arbitrary binary hierarchy tree; and
- **IAS15** as an adaptive, high-accuracy reference.

Ten cases have genuinely branching hierarchies and are expected to favor WHFast-HJ. Three controls have comb-shaped hierarchies that ordinary Jacobi coordinates can represent exactly after a suitable particle ordering. Observed-system cases are **inspired and rescaled benchmarks**, not ephemeris reconstructions.

Scientific background: [WHFast](https://arxiv.org/abs/1506.01084), [IAS15](https://arxiv.org/abs/1409.4779), [planetary stability in hierarchical triples](https://academic.oup.com/mnras/article/382/4/1432/1142027), and [arbitrary nested-binary hierarchies](https://arxiv.org/abs/1511.00944).

> **Runtime:** the default full benchmark uses 100 top-level periods, 600 energy samples, and three timing repeats. It is designed to take roughly 15–30 minutes on the development machine. Set the environment variable `REBOUND_BENCHMARK_QUICK=1` before starting the kernel for a short validation run.

In [ ]:
from dataclasses import dataclass
from pathlib import Path
import itertools
import math
import os
import sys
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Markdown, display

cwd = Path.cwd().resolve()
repo_root = next(
    p for p in (cwd, *cwd.parents)
    if (p / "rebound").is_dir() and (p / "src").is_dir()
)
sys.path.insert(0, str(repo_root))
import rebound

QUICK = os.environ.get("REBOUND_BENCHMARK_QUICK", "0") == "1"
TOP_PERIODS = 0.05 if QUICK else 100.0
N_OUTPUT = 24 if QUICK else 600
TIMING_REPEATS = 1 if QUICK else 3
SCREEN_DIVISORS = (20,) if QUICK else (10, 20, 40, 80)
MAIN_DIVISOR = 20
ENERGY_FLOOR = 5e-16

print("rebound Python:", rebound.__file__)
print("rebound library:", rebound.__libpath__)
print("mode:", "QUICK validation" if QUICK else "FULL benchmark")
print(f"top-level periods={TOP_PERIODS}, outputs={N_OUTPUT}, timing repeats={TIMING_REPEATS}")

## Why the coordinate tree matters

For any internal node joining subsystems $A$ and $B$, hierarchical Jacobi coordinates use

$$
\mathbf r_{AB}=\mathbf R_B-\mathbf R_A,
\qquad
\mu_{AB}=\frac{M_A M_B}{M_A+M_B},
$$

where $\mathbf R_A$ and $\mathbf R_B$ are subsystem barycenters. The dominant two-body Hamiltonian at that node is

$$
H_{AB}^{\rm Kep}
=\frac{\mathbf p_{AB}^2}{2\mu_{AB}}
-\frac{G M_A M_B}{r_{AB}}.
$$

Ordinary WHFast can only form a left-deep chain such as $[[[1,2],3],4]$. Particle reordering can represent any **comb** hierarchy, but no ordering can turn that chain into a branching tree such as $[[1,2],[3,4]]$. WHFast-HJ assigns one Kepler problem to every node of the actual binary tree, leaving a much smaller interaction Hamiltonian when the physical architecture branches.

In [ ]:
@dataclass(frozen=True)
class Leaf:
    name: str
    mass: float


@dataclass(frozen=True)
class OrbitNode:
    name: str
    left: object
    right: object
    a: float
    e: float = 0.0
    inc_deg: float = 0.0
    Omega_deg: float = 0.0
    omega_deg: float = 0.0
    f_deg: float = 0.0


@dataclass(frozen=True)
class Case:
    key: str
    title: str
    category: str
    description: str
    root: object
    expected: str
    source_label: str
    source_url: str
    hj_mode: str = "fixed"


def L(name, mass):
    return Leaf(name, float(mass))


def O(name, left, right, a, e=0.0, inc=0.0, Omega=0.0, omega=0.0, f=0.0):
    return OrbitNode(name, left, right, float(a), float(e), float(inc),
                     float(Omega), float(omega), float(f))


def _shift_particles(particles, target_com):
    shifted = []
    for name, particle in particles:
        p = particle.copy()
        for attr in ("x", "y", "z", "vx", "vy", "vz"):
            setattr(p, attr, getattr(p, attr) + getattr(target_com, attr))
        shifted.append((name, p))
    return shifted


def build_hierarchy(node):
    """Return barycentric leaf particles, mass, name tree, periods, and node metadata."""
    if isinstance(node, Leaf):
        return [(node.name, rebound.Particle(m=node.mass))], node.mass, node.name, [], []

    left_particles, left_mass, left_tree, left_periods, left_meta = build_hierarchy(node.left)
    right_particles, right_mass, right_tree, right_periods, right_meta = build_hierarchy(node.right)

    pair = rebound.Simulation()
    pair.G = 1.0
    pair.add(m=left_mass)
    pair.add(
        m=right_mass, a=node.a, e=node.e,
        inc=np.deg2rad(node.inc_deg), Omega=np.deg2rad(node.Omega_deg),
        omega=np.deg2rad(node.omega_deg), f=np.deg2rad(node.f_deg),
    )
    pair.move_to_com()
    period = 2*np.pi*np.sqrt(node.a**3/(left_mass + right_mass))
    particles = (
        _shift_particles(left_particles, pair.particles[0])
        + _shift_particles(right_particles, pair.particles[1])
    )
    metadata = left_meta + right_meta + [{
        "name": node.name, "a": node.a, "e": node.e,
        "inc_deg": node.inc_deg, "period": period,
    }]
    return (
        particles, left_mass + right_mass, [left_tree, right_tree],
        left_periods + right_periods + [period], metadata,
    )


def leaf_names(node):
    if isinstance(node, Leaf):
        return [node.name]
    return leaf_names(node.left) + leaf_names(node.right)


def hierarchy_orders(node):
    """All depth-first leaf orders obtained by swapping children at every node."""
    if isinstance(node, Leaf):
        return [(node.name,)]
    left = hierarchy_orders(node.left)
    right = hierarchy_orders(node.right)
    orders = [a+b for a in left for b in right] + [b+a for a in left for b in right]
    return list(dict.fromkeys(orders))


def tree_to_indices(tree, order):
    index = {name: i+1 for i, name in enumerate(order)}
    def convert(item):
        return index[item] if isinstance(item, str) else [convert(item[0]), convert(item[1])]
    return convert(tree)


def case_data(case):
    particles, total_mass, name_tree, periods, metadata = build_hierarchy(case.root)
    return {
        "particles": dict(particles), "natural_order": tuple(name for name, _ in particles),
        "total_mass": total_mass, "name_tree": name_tree,
        "periods": np.asarray(periods), "p_min": min(periods), "p_top": periods[-1],
        "metadata": metadata,
    }


def make_simulation(case, order, integrator, divisor=MAIN_DIVISOR):
    data = case_data(case)
    sim = rebound.Simulation()
    sim.G = 1.0
    for name in order:
        sim.add(data["particles"][name])
    sim.move_to_com()
    sim.integrator = integrator
    sim.dt = data["p_min"]/divisor
    if integrator == "ias15":
        sim.integrator.epsilon = 1e-12
    return sim, data, tree_to_indices(data["name_tree"], order)


def relative_energy_error(sim, energy0):
    return max(abs((sim.energy()-energy0)/energy0), ENERGY_FLOOR)


def named_state(sim, order):
    return {
        name: np.array([p.x, p.y, p.z, p.vx, p.vy, p.vz], dtype=float)
        for name, p in zip(order, sim.particles)
    }


def normalized_state_error(state, reference):
    names = sorted(reference)
    a = np.concatenate([state[name] for name in names])
    b = np.concatenate([reference[name] for name in names])
    return np.sqrt(np.mean((a-b)**2))/max(np.sqrt(np.mean(b*b)), np.finfo(float).eps)

## Thirteen architecture categories

All quantities use $G=1$. Adjacent hierarchy levels are separated enough to avoid orbit crossing and close encounters. Inclinations deliberately differ in several cases so the comparison is not restricted to coplanar systems. The first ten trees contain at least one node whose two children are both nontrivial subsystems; the last three are combs.

In [ ]:
CASES = [
    Case(
        "01_hd98800", "Misaligned 2+2 stellar quadruple", "branching stellar quadruple",
        "HD 98800-inspired: two eccentric tight binaries orbit each other on a wider, strongly inclined orbit.",
        O("outer", O("A binary", L("A1",1.1), L("A2",.9),1,.1),
          O("B binary", L("B1",.9), L("B2",.6),1.3,.2,35),18,.15,60),
        "win", "HD 98800 orbital architecture", "https://arxiv.org/abs/2109.02841"),
    Case(
        "02_ph1", "Circumbinary planet in a 2+2 quadruple", "planetary 3+2 branch",
        "PH1/Kepler-64-inspired: a circumbinary planet belongs to one stellar pair while another tight binary forms the other branch.",
        O("outer", O("planetary A", O("A binary",L("A1",1.4),L("A2",.4),1,.2),L("P",.001),4,.03,3),
          O("B binary",L("B1",.8),L("B2",.6),1.2,.1,20),18,.1,15),
        "win", "PH1 discovery", "https://arxiv.org/abs/1210.3612"),
    Case(
        "03_30ari", "S-type planet in a 2+2 quadruple", "planet inside one stellar branch",
        "30 Ari-inspired and mass-amplified: a close S-type companion is nested around one member of a stellar binary, opposite a second stellar binary.",
        O("outer", O("B system",O("host+planet",L("B host",1.2),L("P",.1),.1,.02),L("B companion",.5),.7,.2,8),
          O("A binary",L("A1",.9),L("A2",.7),.2,.1,25),5,.1,20),
        "win", "30 Ari in nested multiple dynamics", "https://academic.oup.com/mnras/article/459/3/2827/2595187"),
    Case(
        "04_castor", "Three-binary sextuple", "three-level stellar sextuple",
        "Castor-inspired and rescaled: two binaries form an inner 2+2 quadruple and a third binary orbits that quadruple.",
        O("outer", O("AB quadruple",O("A binary",L("A1",1.5),L("A2",.5),.5,.05),
          O("B binary",L("B1",1.3),L("B2",.4),.7,.1,15),5,.15,30),
          O("C binary",L("C1",.6),L("C2",.6),.4,.02,50),18,.1,40),
        "win", "Castor sextuple architecture", "https://www.aspbooks.org/a/volumes/article_details/?paper_id=37069"),
    Case(
        "05_double_multiplanet", "Two S-type multiplanet systems", "two planetary branches",
        "A mass-amplified compact analogue of a wide stellar binary whose two components each host a two-planet system.",
        O("stellar orbit",O("A system",O("A inner",L("A",1),L("A p1",.1),.4,.02),L("A p2",.05),1.8,.04,3),
          O("B system",O("B inner",L("B",.8),L("B p1",.08),.45,.03,5),L("B p2",.04),2,.05,8),10,.15,20),
        "win", "Planets in binary-star systems", "https://www.aanda.org/articles/aa/full_html/2026/03/aa57244-25/aa57244-25.html"),
    Case(
        "06_double_moons", "Two planet–moon systems in a stellar binary", "nested satellite branches",
        "Each star hosts a giant planet–moon pair; both moon-scale Kepler problems live below the wide stellar orbit.",
        O("stellar orbit",O("A system",L("A",1),O("A planet-moon",L("A planet",.2),L("A moon",.08),.02,.02,5),.3,.03),
          O("B system",L("B",.8),O("B planet-moon",L("B planet",.16),L("B moon",.064),.02,.03,8),.32,.04,10),1.8,.1,25),
        "win", "Nested-binary hierarchy formalism", "https://arxiv.org/abs/1511.00944"),
    Case(
        "07_triple_planets", "Planet-hosting hierarchical triple", "two occupied triple branches",
        "A mass-amplified compact analogue: an inner binary carries a circumbinary planet while the tertiary carries an S-type planet.",
        O("outer",O("AB+planet",O("AB",L("A",1),L("B",.7),.6,.1),L("AB planet",.05),2,.04,5),
          O("C+planet",L("C",.8),L("C planet",.1),.3,.02,12),8,.1,20),
        "win", "Planetary zones in hierarchical triples", "https://academic.oup.com/mnras/article/382/4/1432/1142027"),
    Case(
        "08_twin_circumbinary", "Twin circumbinary planetary systems", "two planetary triple branches",
        "Two tight stellar binaries each carry a circumbinary planet; the planetary triples orbit one another.",
        O("outer",O("A triple",O("A binary",L("A1",1),L("A2",.6),.8,.1),L("A planet",.001),3.5,.03,4),
          O("B triple",O("B binary",L("B1",.9),L("B2",.5),1,.08,15),L("B planet",.001),4,.04,18),15,.1,25),
        "win", "Arbitrary multiplanet and multistar hierarchies", "https://arxiv.org/abs/1511.00944"),
    Case(
        "09_exomoon_quadruple", "Circumbinary planet–moon plus outer binary", "binary planet branch in a quadruple",
        "A planet–moon binary orbits one stellar binary while a second stellar binary forms the other outer branch.",
        O("outer",O("left",O("A binary",L("A1",1),L("A2",.7),.4,.1),
          O("planet-moon",L("planet",.2),L("moon",.1),.4,.02,5),2.5,.04,8),
          O("B binary",L("B1",.8),L("B2",.6),.4,.1,25),12,.1,20),
        "win", "Nested-binary hierarchy formalism", "https://arxiv.org/abs/1511.00944"),
    Case(
        "10_binary_forest", "Balanced eight-body binary forest", "deep balanced hierarchy",
        "Four tight binaries form two 2+2 quadruples, which then orbit one another; this tests tree depth and traversal cost.",
        O("outer",O("Q1",O("b1",L("a1",1),L("a2",.8),.6,.05),O("b2",L("b1a",.9),L("b2a",.7),.7,.08,10),4,.1,25),
          O("Q2",O("b3",L("c1",.8),L("c2",.6),.8,.06,20),O("b4",L("d1",.7),L("d2",.5),.9,.1,30),4.5,.12,35),42,.1,40),
        "win", "Architecture of hierarchical stellar systems", "https://www.mdpi.com/2218-1997/7/9/352"),
    Case(
        "11_triple_control", "Ordered hierarchical triple", "comb control",
        "The natural tree [[A,B],C] is already ordinary Jacobi form, so fixed-tree HJ should not improve accuracy.",
        O("outer",O("inner",L("A",1),L("B",.7),1,.1),L("C",.4),15,.15,20),
        "control", "Hierarchical triple systems", "https://academic.oup.com/mnras/article/382/4/1432/1142027"),
    Case(
        "12_3plus1_control", "3+1 stellar quadruple", "automatic-tree comb control",
        "A pure comb [[[A,B],C],D] is representable by WHFast ordering; automatic HJ rebuilding adds overhead without changing the split.",
        O("outer",O("triple",O("inner",L("A",1),L("B",.7),1,.1),L("C",.5),6,.1,10),L("D",.3),25,.1,20),
        "control", "3+1 hierarchical multiples", "https://arxiv.org/abs/1511.00944", hj_mode="automatic"),
    Case(
        "13_pluto_control", "Scaled Sun–Pluto–Charon–moon chain", "satellite comb control",
        "A mass-amplified Pluto analogue: Pluto–Charon plus two small circumbinary moons orbit a Sun. Reordering puts the whole hierarchy in one ordinary Jacobi comb.",
        O("heliocentric",L("Sun",1),O("Hydra orbit",O("Nix orbit",O("Pluto-Charon",L("Pluto",1e-3),L("Charon",1.2e-4),.05,.02),
          L("Nix",1e-8),.12,.01),L("Hydra",1e-8),.2,.01),1.2,.2,17),
        "control", "NASA Pluto moon system", "https://science.nasa.gov/dwarf-planets/pluto/moons/facts/"),
]

assert len(CASES) == 13
assert sum(case.expected == "win" for case in CASES) == 10
print("cases:", len(CASES), "expected wins:", sum(c.expected == "win" for c in CASES))

In [ ]:
for number, case in enumerate(CASES, 1):
    data = case_data(case)
    display(Markdown(
        f"### {number}. {case.title}\n\n"
        f"**Category:** {case.category}.  **Expected result:** {case.expected}.  "
        f"**HJ mode:** {case.hj_mode}.\n\n{case.description}\n\n"
        f"Source: [{case.source_label}]({case.source_url})"
    ))
    print("name tree:", data["name_tree"])
    print(f"N={len(data['natural_order'])}, P_top/P_min={data['p_top']/data['p_min']:.2f}")

## Give ordinary WHFast its strongest reasonable ordering

Jacobi coordinates depend on particle order. For each case, the next cell tests every depth-first order obtained by swapping children of the physical hierarchy and selects the order with the smallest short-pilot maximum energy error. This prevents an artificial HJ victory caused only by a poor input order. For branching trees, no candidate can remove the fundamental mismatch; for comb controls, one candidate reproduces the hierarchy exactly.

In [ ]:
def short_error_run(case, order, integrator, divisor=20, horizon=None, samples=20):
    sim, data, tree = make_simulation(case, order, integrator, divisor)
    if horizon is None:
        horizon = min(0.05*data["p_top"], 50*data["p_min"])
    total_steps = max(1, int(np.ceil(horizon/sim.dt)))
    marks = np.unique(np.linspace(1, total_steps, samples, dtype=int))
    energy0 = sim.energy()
    if integrator == "whfast_hj" and case.hj_mode == "fixed":
        sim.integrate(sim.t, given_tree=True, tree=tree)
    previous = 0
    errors = []
    start = time.perf_counter()
    with warnings.catch_warnings():
        warnings.simplefilter("error")
        for mark in marks:
            sim.steps(int(mark-previous))
            previous = int(mark)
            error = relative_energy_error(sim, energy0)
            if not np.isfinite(error):
                raise FloatingPointError(f"non-finite energy in {case.key}/{integrator}")
            errors.append(error)
    return max(errors), time.perf_counter()-start


def optimize_whfast_order(case):
    candidates = hierarchy_orders(case.root)
    scored = []
    for order in candidates:
        error, elapsed = short_error_run(case, order, "whfast")
        scored.append((error, elapsed, order))
    scored.sort(key=lambda row: row[0])
    return scored[0][2], scored


OPTIMAL_ORDERS = {}
ordering_start = time.perf_counter()
for case in CASES:
    best_order, scores = optimize_whfast_order(case)
    OPTIMAL_ORDERS[case.key] = best_order
    print(f"{case.key:24s} candidates={len(scores):3d} best={best_order} pilot_error={scores[0][0]:.3e}")
print(f"ordering pilots completed in {time.perf_counter()-ordering_start:.2f} s")

## Timestep pre-screen

The full run checks $\Delta t=P_{\min}/d$ for $d\in\{10,20,40,80\}$ over a short horizon. A same-step candidate passes the primary HJ criterion when

$$
\frac{\max |\Delta E/E_0|_{\rm WHFast}}
{\max |\Delta E/E_0|_{\rm HJ}}\ge 10
$$

and the short HJ runtime is no more than three times the WHFast runtime. Quick mode checks only $d=20$. The final classification is recomputed from the long trajectory and independent timing runs.

In [ ]:
SCREEN = []
for case in CASES[:10]:
    data = case_data(case)
    horizon = min(0.10*data["p_top"], 200*data["p_min"])
    for divisor in SCREEN_DIVISORS:
        order = OPTIMAL_ORDERS[case.key]
        err_w, sec_w = short_error_run(case, order, "whfast", divisor, horizon, 32)
        err_h, sec_h = short_error_run(case, order, "whfast_hj", divisor, horizon, 32)
        ratio = err_w/err_h
        passed = ratio >= 10 and sec_h/sec_w <= 3
        SCREEN.append({"case":case.key, "divisor":divisor, "whfast":err_w,
                       "hj":err_h, "error_ratio":ratio, "runtime_ratio":sec_h/sec_w,
                       "passed":passed})
        print(f"{case.key:24s} P/{divisor:<2d} error ratio={ratio:9.2f} "
              f"runtime ratio={sec_h/sec_w:5.2f} {'PASS' if passed else 'review'}")

## Long energy histories and independent timing

The main symplectic runs use $\Delta t=P_{\min}/20$. Fixed HJ trees are installed once before stepping, so parsing is excluded from integration time. The automatic-tree control rebuilds its hierarchy inside every HJ step. For each integrator, the sampled full trajectory is the unmeasured warm-up; runtime measurements then use three fresh one-shot simulations and do not include setup, plotting, or energy sampling.

In [ ]:
def trajectory(case, integrator, order):
    sim, data, tree = make_simulation(case, order, integrator)
    sampling_dt = sim.dt
    total_steps = max(1, int(np.ceil(TOP_PERIODS*data["p_top"]/sampling_dt)))
    marks = np.unique(np.linspace(1, total_steps, N_OUTPUT, dtype=int))
    energy0 = sim.energy()
    if integrator == "whfast_hj" and case.hj_mode == "fixed":
        sim.integrate(sim.t, given_tree=True, tree=tree)
    times, errors = [], []
    previous = 0
    start = time.perf_counter()
    with warnings.catch_warnings():
        warnings.simplefilter("error")
        for mark in marks:
            target = mark*sampling_dt
            if integrator == "ias15":
                sim.integrate(target, exact_finish_time=1)
            else:
                sim.steps(int(mark-previous))
            previous = int(mark)
            times.append(sim.t/data["p_top"])
            errors.append(relative_energy_error(sim, energy0))
    values = np.asarray(errors)
    if not np.all(np.isfinite(values)):
        raise FloatingPointError(f"non-finite trajectory in {case.key}/{integrator}")
    com = sim.com()
    com_error = float(np.linalg.norm([com.x, com.y, com.z, com.vx, com.vy, com.vz]))
    if not np.isfinite(com_error) or com_error > 1e-9:
        raise AssertionError(f"center-of-mass drift in {case.key}/{integrator}: {com_error}")
    return {"time":np.asarray(times), "energy_error":values,
            "max_error":float(values.max()), "rms_error":float(np.sqrt(np.mean(values**2))),
            "state":named_state(sim, order), "trajectory_seconds":time.perf_counter()-start,
            "steps":total_steps, "com_error":com_error}


def timed_integration(case, integrator, order):
    durations = []
    for repeat in range(TIMING_REPEATS):
        sim, data, tree = make_simulation(case, order, integrator)
        total_steps = max(1, int(np.ceil(TOP_PERIODS*data["p_top"]/sim.dt)))
        target = total_steps*sim.dt
        if integrator == "whfast_hj" and case.hj_mode == "fixed":
            sim.integrate(sim.t, given_tree=True, tree=tree)
        start = time.perf_counter_ns()
        if integrator == "ias15":
            sim.integrate(target, exact_finish_time=1)
        else:
            sim.steps(total_steps)
        durations.append((time.perf_counter_ns()-start)*1e-9)
    return float(np.median(durations)), durations


RESULTS = {}
benchmark_start = time.perf_counter()
for number, case in enumerate(CASES, 1):
    order = OPTIMAL_ORDERS[case.key]
    print(f"[{number:02d}/13] {case.title}", flush=True)
    RESULTS[case.key] = {"case":case, "order":order}
    for integrator in ("whfast", "whfast_hj", "ias15"):
        run = trajectory(case, integrator, order)
        median_seconds, repeats = timed_integration(case, integrator, order)
        run["runtime"] = median_seconds
        run["runtime_repeats"] = repeats
        RESULTS[case.key][integrator] = run
        print(f"  {integrator:9s} max|dE/E|={run['max_error']:.3e} "
              f"median={median_seconds:.3f}s", flush=True)
    reference = RESULTS[case.key]["ias15"]["state"]
    for integrator in ("whfast", "whfast_hj", "ias15"):
        RESULTS[case.key][integrator]["state_error"] = normalized_state_error(
            RESULTS[case.key][integrator]["state"], reference)
TOTAL_BENCHMARK_SECONDS = time.perf_counter()-benchmark_start
print(f"benchmark cells completed in {TOTAL_BENCHMARK_SECONDS/60:.2f} minutes")

In [ ]:
COLORS = {"whfast":"#d95f02", "whfast_hj":"#1b9e77", "ias15":"#3b5bdb"}
LABELS = {"whfast":"WHFast", "whfast_hj":"WHFast-HJ", "ias15":"IAS15"}

for number, case in enumerate(CASES, 1):
    result = RESULTS[case.key]
    display(Markdown(
        f"## Case {number}: {case.title}\n\n"
        f"**Category:** {case.category}. {case.description} "
        f"[Source: {case.source_label}]({case.source_url})."
    ))
    fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.2), gridspec_kw={"width_ratios":[2.2,1]})
    for integrator in ("whfast", "whfast_hj", "ias15"):
        run = result[integrator]
        axes[0].plot(run["time"], run["energy_error"], color=COLORS[integrator],
                     label=LABELS[integrator], lw=1.2)
    axes[0].set_yscale("log")
    axes[0].set_xlabel(r"$t/P_{\rm top}$")
    axes[0].set_ylabel(r"$|[E(t)-E(0)]/E(0)|$")
    axes[0].set_title("Relative energy error")
    axes[0].grid(alpha=.25)
    axes[0].legend()

    integrators = ("whfast", "whfast_hj", "ias15")
    runtimes = [result[name]["runtime"] for name in integrators]
    axes[1].bar([LABELS[name] for name in integrators], runtimes,
                color=[COLORS[name] for name in integrators])
    axes[1].set_yscale("log")
    axes[1].set_ylabel("median integration time [s]")
    axes[1].set_title("Runtime (setup excluded)")
    axes[1].tick_params(axis="x", rotation=25)
    axes[1].grid(axis="y", alpha=.25)
    fig.suptitle(f"{number}. {case.title}", fontsize=13)
    fig.tight_layout()
    plt.show()

## Summary and measured classification

Energy conservation is the requested primary metric, but it is not sufficient by itself: symplectic energy error can remain bounded while orbital phase diverges. The table therefore also reports a normalized final six-dimensional state difference relative to IAS15. A case is labeled a measured HJ win only when the long-run energy ratio is at least ten and HJ costs no more than three times WHFast. Any disagreement with the expected label is retained rather than hidden.

In [ ]:
SUMMARY = []
header = (f"{'case':24s} {'expected':9s} {'measured':10s} {'err W/HJ':>10s} "
          f"{'time HJ/W':>10s} {'state W':>10s} {'state HJ':>10s}")
print(header)
print("-"*len(header))
for case in CASES:
    result = RESULTS[case.key]
    error_ratio = result["whfast"]["max_error"]/result["whfast_hj"]["max_error"]
    runtime_ratio = result["whfast_hj"]["runtime"]/result["whfast"]["runtime"]
    measured = "HJ win" if error_ratio >= 10 and runtime_ratio <= 3 else "not worth"
    row = {"case":case.key, "title":case.title, "expected":case.expected,
           "measured":measured, "error_ratio":error_ratio, "runtime_ratio":runtime_ratio,
           "whfast_error":result["whfast"]["max_error"],
           "hj_error":result["whfast_hj"]["max_error"],
           "ias15_error":result["ias15"]["max_error"],
           "whfast_rms":result["whfast"]["rms_error"],
           "hj_rms":result["whfast_hj"]["rms_error"],
           "ias15_rms":result["ias15"]["rms_error"],
           "whfast_runtime":result["whfast"]["runtime"],
           "hj_runtime":result["whfast_hj"]["runtime"],
           "ias15_runtime":result["ias15"]["runtime"],
           "whfast_state_error":result["whfast"]["state_error"],
           "hj_state_error":result["whfast_hj"]["state_error"]}
    SUMMARY.append(row)
    print(f"{case.key:24s} {case.expected:9s} {measured:10s} {error_ratio:10.2f} "
          f"{runtime_ratio:10.2f} {row['whfast_state_error']:10.2e} {row['hj_state_error']:10.2e}")

print("\nDetailed error and runtime table")
detail_header = (f"{'case':24s} {'W max':>10s} {'W rms':>10s} {'W sec':>8s} "
                 f"{'HJ max':>10s} {'HJ rms':>10s} {'HJ sec':>8s} "
                 f"{'I max':>10s} {'I rms':>10s} {'I sec':>8s}")
print(detail_header)
print("-"*len(detail_header))
for row in SUMMARY:
    print(f"{row['case']:24s} {row['whfast_error']:10.2e} {row['whfast_rms']:10.2e} "
          f"{row['whfast_runtime']:8.3f} {row['hj_error']:10.2e} {row['hj_rms']:10.2e} "
          f"{row['hj_runtime']:8.3f} {row['ias15_error']:10.2e} {row['ias15_rms']:10.2e} "
          f"{row['ias15_runtime']:8.3f}")

labels = [row["case"].replace("_", " ") for row in SUMMARY]
matrix = np.array([[np.log10(max(row["error_ratio"], 1e-12)),
                    np.log10(max(row["runtime_ratio"], 1e-12))] for row in SUMMARY])
fig, ax = plt.subplots(figsize=(7.5, 7.2))
image = ax.imshow(matrix, aspect="auto", cmap="coolwarm")
ax.set_xticks([0,1], [r"$\log_{10}$(WHFast error / HJ error)",
                       r"$\log_{10}$(HJ time / WHFast time)"], rotation=12)
ax.set_yticks(np.arange(len(labels)), labels)
for i in range(matrix.shape[0]):
    for j in range(matrix.shape[1]):
        ax.text(j, i, f"{matrix[i,j]:.2f}", ha="center", va="center", fontsize=8)
fig.colorbar(image, ax=ax, label="log-ratio")
ax.set_title("Accuracy advantage and runtime cost")
fig.tight_layout()
plt.show()

measured_wins = sum(row["measured"] == "HJ win" for row in SUMMARY[:10])
measured_controls = sum(row["measured"] == "not worth" for row in SUMMARY[10:])
print(f"confirmed wins among first ten: {measured_wins}/10")
print(f"controls where HJ is not worthwhile: {measured_controls}/3")
print(f"total benchmark runtime: {TOTAL_BENCHMARK_SECONDS/60:.2f} min")

## Interpretation and limitations

- A large HJ advantage is expected when the physical system contains **two or more simultaneously important branches**. The tree places each tight binary, planet–moon pair, or planetary subsystem inside its own Kepler Hamiltonian.
- A simple triple, a 3+1 chain, or the reordered Pluto control is already a Jacobi comb. In these cases WHFast-HJ performs essentially the same split with additional tree traversal—and, for the automatic case, tree-rebuilding—overhead.
- IAS15 is adaptive and not symplectic. Its nearly machine-level energy errors provide a useful finite-time reference, while WHFast methods are intended for inexpensive long integrations with bounded error.
- These tests deliberately exclude close encounters and hierarchy changes. The fixed-step Stumpff overflow diagnosed in `0727_debug.ipynb` can affect both WHFast variants when an encounter is unresolved.
- Total relative energy can be dominated by the tightest binary. The IAS15-referenced state-error columns are included to expose cases where apparently good energy conservation hides phase error.
- Runtime conclusions apply to this REBOUND implementation, compiler, and machine. Fixed-tree parsing and all Python plotting/sampling work are excluded; automatic hierarchy discovery is included where explicitly selected.